# 00 — Ingestão

Baixa o dataset `clicks-conversion-tracking` do Kaggle, valida o arquivo e salva uma cópia limpa em CSV para os notebooks seguintes.

---

## Antes de rodar: configurar a chave do Kaggle

**Passo 1** — Acesse [kaggle.com](https://www.kaggle.com) → foto de perfil → **Settings** → seção **API** → **Create New Token**. Salve o arquivo `kaggle.json` que aparecer.

**Passo 2** — Coloque o arquivo em:
- **Windows:** `C:\Users\SeuUsuario\.kaggle\kaggle.json`
- **Mac/Linux:** `~/.kaggle/kaggle.json`

Para criar a pasta no Windows (PowerShell):
```powershell
New-Item -ItemType Directory -Force -Path "$env:USERPROFILE\.kaggle"
```

**Passo 3** — Rode as células normalmente.

In [19]:
from pathlib import Path
import pandas as pd


In [20]:
RAW       = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

DATASET   = "loveall/clicks-conversion-tracking"
CSV_RAW   = RAW / "KAG_conversion_data.csv"
CSV_CLEAN = RAW / "conversion_data.csv"

## 1. Verificar chave do Kaggle

In [21]:
kaggle_json = Path.home() / ".kaggle" / "kaggle.json"

if kaggle_json.exists():
    print(f"Chave encontrada em: {kaggle_json}")
else:
    print("ATENÇÃO: kaggle.json não encontrado.")
    print(f"Esperado em: {kaggle_json}")
    print("Siga as instruções no topo deste notebook antes de continuar.")

ATENÇÃO: kaggle.json não encontrado.
Esperado em: C:\Users\samue\.kaggle\kaggle.json
Siga as instruções no topo deste notebook antes de continuar.


## 2. Download do dataset

Se o CSV já estiver em `data/raw/`, esta célula apenas informa e segue adiante.

In [22]:
if CSV_RAW.exists():
    print(f"Arquivo já existe: {CSV_RAW}")
else:
    print("Baixando dataset do Kaggle...")
    kaggle.api.authenticate()
    kaggle.api.dataset_download_files(DATASET, path=str(RAW), unzip=True)
    print(f"Download concluído: {CSV_RAW}")

Arquivo já existe: ..\data\raw\KAG_conversion_data.csv


## 3. Validação básica

In [23]:
df = pd.read_csv(CSV_RAW)

print(f"Linhas    : {len(df)}")
print(f"Colunas   : {list(df.columns)}")
print(f"Nulos     : {df.isnull().sum().sum()}")
print(f"Duplicados: {df.duplicated().sum()}")
df.head()

Linhas    : 1143
Colunas   : ['ad_id', 'xyz_campaign_id', 'fb_campaign_id', 'age', 'gender', 'interest', 'Impressions', 'Clicks', 'Spent', 'Total_Conversion', 'Approved_Conversion']
Nulos     : 0
Duplicados: 0


,ad_id,xyz_campaign_id,fb_campaign_id,age,gender,interest,Impressions,Clicks,Spent,Total_Conversion,Approved_Conversion
0,708746,916,103916,30-34,M,15,7350,1,1.43,2,1
1,708749,916,103917,30-34,M,16,17861,2,1.82,2,0
2,708771,916,103920,30-34,M,20,693,0,0.00,1,0
3,708815,916,103928,30-34,M,28,4259,1,1.25,1,0
4,708818,916,103928,30-34,M,28,4133,1,1.29,1,1


In [24]:
print(df.dtypes)
df.describe()

ad_id                    int64
xyz_campaign_id          int64
fb_campaign_id           int64
age                        str
gender                     str
interest                 int64
Impressions              int64
Clicks                   int64
Spent                  float64
Total_Conversion         int64
Approved_Conversion      int64
dtype: object


,ad_id,xyz_campaign_id,fb_campaign_id,interest,Impressions,Clicks,Spent,Total_Conversion,Approved_Conversion
count,1.143000e+03,1143.000000,1143.000000,1143.000000,1.143000e+03,1143.000000,1143.000000,1143.000000,1143.000000
mean,9.872611e+05,1067.382327,133783.989501,32.766404,1.867321e+05,33.390201,51.360656,2.855643,0.944007
std,1.939928e+05,121.629393,20500.308622,26.952131,3.127622e+05,56.892438,86.908418,4.483593,1.737708
min,7.087460e+05,916.000000,103916.000000,2.000000,8.700000e+01,0.000000,0.000000,0.000000,0.000000
25%,7.776325e+05,936.000000,115716.000000,16.000000,6.503500e+03,1.000000,1.480000,1.000000,0.000000
50%,1.121185e+06,1178.000000,144549.000000,25.000000,5.150900e+04,8.000000,12.370000,1.000000,1.000000
75%,1.121804e+06,1178.000000,144657.500000,31.000000,2.217690e+05,37.500000,60.025000,3.000000,1.000000
max,1.314415e+06,1178.000000,179982.000000,114.000000,3.052003e+06,421.000000,639.949998,60.000000,21.000000


## 4. Padronização de nomes de colunas

Converte para snake_case para que todos os notebooks e módulos usem os mesmos nomes.

In [25]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Colunas após padronização:")
print(list(df.columns))

Colunas após padronização:
['ad_id', 'xyz_campaign_id', 'fb_campaign_id', 'age', 'gender', 'interest', 'impressions', 'clicks', 'spent', 'total_conversion', 'approved_conversion']


## 5. Salvar CSV limpo

A partir daqui os outros notebooks leem `conversion_data.csv`, não o arquivo bruto original.

In [26]:
df.to_csv(CSV_CLEAN, index=False)
print(f"Salvo em: {CSV_CLEAN}")
print(f"Tamanho : {CSV_CLEAN.stat().st_size / 1024:.1f} KB")

Salvo em: ..\data\raw\conversion_data.csv
Tamanho : 60.6 KB


## 6. Verificação final

In [27]:
df_check = pd.read_csv(CSV_CLEAN)
assert len(df_check) == len(df), "Contagem de linhas diverge após salvar."
print("OK — arquivo íntegro.")
df_check.head(3)

OK — arquivo íntegro.


,ad_id,xyz_campaign_id,fb_campaign_id,age,gender,interest,impressions,clicks,spent,total_conversion,approved_conversion
0,708746,916,103916,30-34,M,15,7350,1,1.43,2,1
1,708749,916,103917,30-34,M,16,17861,2,1.82,2,0
2,708771,916,103920,30-34,M,20,693,0,0.00,1,0
